In [0]:
source_dir="/Volumes/eccom_catalog/default/ecomm_data/shipping_data/Source/"
archive_dir="/Volumes/eccom_catalog/default/ecomm_data/shipping_data/Archive/"
shipping_stage_table="eccom_catalog.default.shipping_stage"
error_table="eccom_catalog.default.error_table"

In [0]:
#Read shipping csv file
from pyspark.sql.functions import col,current_timestamp,current_date,lit,datediff,to_date,when
from datetime import datetime
try:
	df_shipping=spark.read.csv(source_dir, header=True, inferSchema=True, dateFormat="yyyy-MM-dd", timestampFormat="yyyy-MM-dd HH:mm:ss")
	df_shipping=df_shipping.withColumn("processed_timestamp", current_timestamp()).withColumn("batch_id", lit(datetime.now().strftime("%Y%m%d_%H%M%S"))).withColumn("source_system", lit("ecommerce_shipping"))
	print("Successfully read shipping file")

except Exception as e:
	print(f"Failed to read shipping file: {str(e)}")
	raise

In [0]:
#Data Quality checks
try:
	total_records=df_shipping.count()
	total_null_shipping_ids=df_shipping.filter(col("shipping_id").isNull()).count()
	total_invalid_shipping_costs=df_shipping.filter(col("shipping_cost")<0).count()
	total_negative_weight=df_shipping.filter(col("package_weight")<=0).count()
	total_invalid_insurance=df_shipping.filter(col("insurance_value")<=0).count()
	
	print(f"Total records: {total_records}")
	print(f"Total null shipping ids: {total_null_shipping_ids}")
	print(f"Total invalid shipping costs: {total_invalid_shipping_costs}")
	print(f"Total negative product weight: {total_negative_weight}")
	print(f"Total invalid insurance: {total_invalid_insurance}")
	
	valid_shipping=df_shipping.filter((col("shipping_id").isNotNull()) & (col("shipping_cost")>=0) & (col("package_weight")>0) & (col ("insurance_value")>0))
	invalid_shipping=df_shipping.filter((col("shipping_id").isNull()) | (col("shipping_cost")<0) | (col("package_weight")<=0) | (col ("insurance_value")<=0))
	total_valid_records=valid_shipping.count()
	total_invalid_records=invalid_shipping.count()
	print(f"Total valid records: {total_valid_records}")
	print(f"Total invalid records: {total_invalid_records}")
	print("Successfully applied data quality checks")
	
except Exception as e:
	print(f"Failed to apply data quality checks: {str(e)}")
	raise

In [0]:
#Data enrichment - shipping analytics
try:
	#Calculate delivery performance metrices
	df_shipping_delivery=valid_shipping.withColumn("delivery_days", when(col("actual_delivery").isNotNull(), datediff(col("actual_delivery"), to_date(col("created_timestamp")))).otherwise(None))	
																   
	#Calculate estimated delivery days
	df_shipping_delivery=df_shipping_delivery.withColumn("estimated_delivery_days", when(col("estimated_delivery").isNotNull(), datediff(col("estimated_delivery"),to_date(col("created_timestamp")))))
	#Calculate delivery performance
	df_shipping_delivery=df_shipping_delivery.withColumn("delivery_performance", when(col("actual_delivery").isNull(), "Pending")
																				 .when(col("delivery_days")<=col("estimated_delivery_days"),"On Time") 
																				 .when(col("delivery_days")<=(col("estimated_delivery_days")+1), "Slightly Delayed") 
																				 .otherwise("Delayed"))
	#Create shipping cost category
	df_shipping_delivery=df_shipping_delivery.withColumn("cost_category", when(col("shipping_cost")<10, "Low Cost") 
																		  .when(col("shipping_cost")<20, "Medium Cost")
																		  .otherwise("High Cost"))	
	#Calculate cost per weight ratio
	df_shipping_delivery=df_shipping_delivery.withColumn("cost_per_kg", col("shipping_cost")/col("package_weight"))
	print("Data enrichment completed")

except Exception as e:
	print(f"Unsuccessfull data enrichment: {str(e)}")
	raise	

In [0]:
try:
	#Write valid data to staging table
	df_shipping_delivery.write.format("delta").mode("overwrite").saveAsTable(shipping_stage_table)	
	print("Successfully write data to shipping stage table")
	if total_invalid_records>0:
		#Write invalid shipping in error table
		invalid_shipping.withColumn("error_reason", lit("Data validation fail")).withColumn("error_timestamp", current_timestamp())\
		.write.format("delta").mode("append").saveAsTable(error_table)
		print(f"Logged {total_invalid_records} invalid records to error table")
	else:
		print("No invalid shipping found")

except Exception as e:
	print(f"Failed to write data: {str(e)}")
	raise

In [0]:
#Archived processed files
try:
	files=dbutils.fs.ls(source_dir)
	archive_count=0
	for file in files:
		if file.name.endswith(".csv"):
			source=file.path
			archive=archive_dir+file.name
			dbutils.fs.mv(source,archive)
			archive_count+=1
		print(f"Archived file: {file.name}")
	print(f"Successfully archive {archive_count} files")

except Exception as e:
	print(f"Error in archiving files: {str(e)}")
	raise

In [0]:
#Processing log summary
import json
try:
	processing_summary={
	"task": "shipping_stage_load",
	"timestamp": datetime.now().isoformat(),
	"total_records": total_records,
	"valid_records": total_valid_records,
	"invalid_records": total_invalid_records,
	"archived_files": archive_count,
	"status": "SUCCESS" if total_invalid_records==0 else "SUCCESS_WITH_WARNINGS"
	}
	print(f"Processing summary: {json.dumps(processing_summary)}")

	df_processing=spark.createDataFrame([processing_summary])
	df_processing.write.format("delta").mode("append").saveAsTable("eccom_catalog.default.processing_log")
	print("Successfully write in processing log table")

except Exception as e:
	print(f"Failed to write in processing log: {str(e)}")
	raise